# Build a Transformer from Scratch

## The Concept

The architecture, annotated:

```
input tokens (B, N)
   │
   ▼
token embedding + positional embedding  ◀── Lesson 04 (RoPE option)
   │
   ▼
┌──── block × L ────────────────────┐
│  RMSNorm                          │  ◀── Lesson 05
│  MultiHeadAttention (causal)      │  ◀── Lesson 03 + 07 (causal mask)
│  residual                         │
│  RMSNorm                          │
│  SwiGLU FFN                       │  ◀── Lesson 05
│  residual                         │
└────────────────────────────────── ┘
   │
   ▼
final RMSNorm
   │
   ▼
lm_head (tied to token embedding)
   │
   ▼
logits (B, N, V)
   │
   ▼
shift-by-one cross-entropy            ◀── Lesson 07
```


# MVP Version

In [17]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)

from pathlib import Path

corpus_path = Path("temp/tinyshakespeare.txt")

with open(corpus_path, "r") as f:
    corpus = f.read()
    # print(corpus[:1000])

# Data Preparation
chars = sorted(set(corpus))

vocab_size = len(chars)

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# Configuration

block_size = 64
d_model = 64
n_heads = 4
n_layers = 3
ffn_expansion = 2.67
batch_size = 32
max_steps = 3000
eval_interval = 500
lr = 3e-4
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Model

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)

class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, block_size):
        super().__init__()
        assert d % h == 0
        self.h = h
        self.d_head = d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        BATCH_ALIGN = 1
        HEAD_ALIGN = 1
        self.register_buffer("mask",
            torch.tril(torch.ones(block_size, block_size)).view(BATCH_ALIGN, HEAD_ALIGN, block_size, block_size)
        )

    def forward(self, x):
        B, N, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q = q.view(B, N, self.h, self.d_head).transpose(1, 2)
        k = k.view(B, N, self.h, self.d_head).transpose(1, 2)
        v = v.view(B, N, self.h, self.d_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_head))
        att = att.masked_fill(self.mask[:, :, :N, :N] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, N, D)   
        return self.out(y)     

class SwiGLUFFN(nn.Module):
    def __init__(self, d, expansion):
        super().__init__()
        h = int(d * expansion)
        self.w1 = nn.Linear(d, h, bias=False)
        self.w2 = nn.Linear(h, d, bias=False)
        self.w3 = nn.Linear(d, h, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))
    
class Block(nn.Module):
    def __init__(self, d, h, block_size, expansion):
        super().__init__()
        self.ln1 = RMSNorm(d)
        self.attn = CausalSelfAttention(d, h, block_size)
        self.ln2 = RMSNorm(d)
        self.ff = SwiGLUFFN(d, expansion)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, d, h, n_layers, block_size, expansion):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(block_size, d)
        self.blocks = nn.ModuleList(
            [Block(d, h, block_size, expansion) for _ in range(n_layers)]
        )
        self.norm_f = RMSNorm(d)
        self.lm_head = nn.Linear(d, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight # Tied
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, N = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(N, device=idx.device))
        x = tok + pos
        for b in self.blocks:
            x = b(x)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([src[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

model = GPT(vocab_size, d_model, n_heads, n_layers, block_size, ffn_expansion).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"=== capstone transformer ===")
print(f"device:        {device}")
print(f"vocab_size:    {vocab_size}")
print(f"block_size:    {block_size}")
print(f"d_model:       {d_model}")
print(f"n_heads:       {n_heads}")
print(f"n_layers:      {n_layers}")
print(f"parameters:    {n_params}")
print()

opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)

print(f"Training for {max_steps} steps...")

for step in range(max_steps + 1):
    if step % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            x, y = get_batch("train")
            _, train_loss = model(x, y)
            x, y = get_batch("val")
            _, val_loss = model(x, y)

        model.train()
        print(f"step {step}: train loss {train_loss:.4f}, val loss {val_loss:.4f}")

    if step == max_steps:
        break

    x, y = get_batch("train")
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()



print()
print("=== sample ===")
context = "First Citizen:\n"
prompt = torch.tensor([[stoi[c] for c in context]], dtype=torch.long, device=device)
out = model.generate(prompt, max_new_tokens=300, temperature=0.8, top_k=10)
sampled = "".join(itos[i] for i in out[0].tolist())
print(sampled)
    

=== capstone transformer ===
device:        mps
vocab_size:    65
block_size:    64
d_model:       64
n_heads:       4
n_layers:      3
parameters:    155776

Training for 3000 steps...
step 0: train loss 42.5946, val loss 41.9231
step 500: train loss 2.5963, val loss 2.6834
step 1000: train loss 2.4906, val loss 2.4828
step 1500: train loss 2.3691, val loss 2.3928
step 2000: train loss 2.3222, val loss 2.3274
step 2500: train loss 2.3012, val loss 2.2473
step 3000: train loss 2.2427, val loss 2.2305

=== sample ===
First Citizen:
I'le borty alound saing san mones heand of offad wild.

HORDIICH:
Mane to my, sile or the dand maithte,

Winther, stouge in ime wam weell my cored: al hen stromay
and youg thing hing mom the ashuck meallld and,
Thus dell cour thin dand oure aid, meartingedy,

Thas nole ath of or inggr and my wile in
